# Lecția 18: Protejarea agenților AI cu chitanțe criptografice

## Caiet practic

Acest caiet parcurge patru exerciții:

1. **Semnați prima chitanță** pentru un apel de instrument al agentului și verificați-o.
2. **Modificați chitanța** și observați eșecul verificării.
3. **Construiți un lanț de trei chitanțe** și confirmați integritatea lanțului.
4. **Încapsulați un apel de instrument din Microsoft Agent Framework** astfel încât fiecare acțiune să emită o chitanță.

Toate primitivele criptografice sunt importate din biblioteci bine întreținute (`pynacl` pentru Ed25519, `jcs` pentru JSON canonic RFC 8785, `hashlib` din biblioteca standard Python pentru SHA-256). Logica chitanței în sine este cod Python simplu pe care îl puteți citi și modifica.

Rulați celulele în ordine. Fiecare secțiune este scurtă și autonomă.


## Setup

Instalați cele două dependențe. Ambele au licențe permisive (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Utilitare Ajutătoare

Acești doi ajutători gestionează codificarea base64url (fără completare) și hashing-ul SHA-256 al obiectelor arbitrare. Ei păstrează restul caietului concentrat pe logica chitanței propriu-zise.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: Semnează primul tău chitanță

Imaginează-ți că agentul nostru pentru **Contoso Travel** tocmai a căutat zboruri de la Sydney la Los Angeles pentru un client. Dorim să înregistrăm această apelare a instrumentului ca o chitanță semnată, astfel încât un auditor viitor să o poată verifica fără să ne aibă încredere.

### Pasul 1.1: Generează o cheie de semnare

În producție, cheia de semnare a agentului ar fi stocată într-un modul hardware de securitate (HSM), Azure Key Vault sau într-un depozit protejat similar. Pentru această lecție generăm o cheie proaspătă în memorie.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: Construiește sarcina de nivel pentru chitanță

Sarcina de nivel conține tot ce dorim ca chitanța să ateste: cine a acționat, ce instrument, cu ce argumente, ce s-a întors, sub ce politică și când. Hash-uim argumentele și rezultatul în loc să le includem inline pentru ca chitanța să nu divulge conținut sensibil.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Pasul 1.3: Semnează și asamblează chitanța

Trei pași:

1. Canonicalizează payload-ul folosind JCS astfel încât două implementări care produc aceeași chitanță logică să producă octeți identici la nivel de byte.
2. Calculează hash-ul octeților canonicali cu SHA-256.
3. Semnează hash-ul cu cheia privată Ed25519.

Semnătura este apoi atașată la payload-ul original pentru a produce chitanța finală.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Pasul 1.4: Verificarea chitanței

Verificarea inversează procesul. Eliminăm semnătura, recalculăm hash-ul canonic și verificăm semnătura cu cheia publică din chitanță.

Un auditor care face această verificare nu are nevoie de nimic de la noi, cu excepția chitanței în sine. Nicio apelare a unui serviciu, nicio interogare a unui director de chei, niciun fel de încredere necesară.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Ar trebui să vedeți `Receipt is valid: True`. Agentul a generat prima înregistrare de audit semnată criptografic.


## Secțiunea 2: Modifică chitanța

Scopul chitanțelor este să fie evident când sunt modificate. Să demonstrăm acest lucru.

Vom modifica exact un caracter din chitanță și vom urmări cum verificarea eșuează.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Ce s-a întâmplat?

Când am schimbat `policy_id`, octeții canonici s-au schimbat. Hash-ul SHA-256 al acelor octeți s-a schimbat. Semnătura (care era peste hash-ul original) nu mai corespunde noului hash. Verificarea întoarce corect `False`.

Nu există nicio metodă de a modifica vreun câmp al chitanței și totuși să fie validă verificarea, cu excepția cazului în care atacatorul deține cheia privată. Atâta timp cât cheia privată este într-un seif de chei și cheia publică este publicată, manipularea este imposibil de ascuns.

Încearcă și tu: modifică `tool_name` sau `agent_id` sau `timestamp` în celula de mai sus și reexecută. Fiecare modificare produce o chitanță invalidă.


## Secțiunea 3: Legarea bonurilor împreună

Un singur bon protejează o acțiune. Majoritatea agenților efectuează multe acțiuni. Pentru a face întreaga secvență evidentă pentru manipulare, legăm fiecare bon de cel precedent prin includerea hash-ului bonului anterior în conținutul noului bon.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Dacă cineva elimină sau reordonează un bon, lanțul se întrerupe exact în acel punct. Verificarea oricărui bon ulterior eșuează deoarece `previous_receipt_hash` nu mai corespunde cu hash-ul real al predecesorului său.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Acum rupe lanțul prin manipularea chitanței din mijloc și verifică din nou. Chitanța manipulată eșuează verificarea semnăturii, ȘI următoarea chitanță eșuează verificarea legăturii lanțului (deoarece `previous_receipt_hash` nu mai corespunde cu hash-ul chitanței modificate din mijloc).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Chitanța 0 încă se verifică (nu a fost modificată și nu are un predecesor de care să depindă). Chitanța 1 eșuează verificarea semnăturii deoarece am schimbat `tool_args_hash`. Chitanța 2 eșuează verificarea lanțului de legături deoarece `previous_receipt_hash` a fost calculat în raport cu chitanța originală 1 (care acum este modificată).

Chiar dacă un atacator re-semnează chitanța 1 modificată (ceea ce nu poate face fără cheia privată), nepotrivirea lanțului de legături din chitanța 2 ar expune totuși modificarea. Pentru a ascunde schimbarea, atacatorul ar trebui să re-semneze fiecare chitanță începând de la punctul modificării spre înainte, ceea ce necesită posesia cheii private.


## Secțiunea 4: Învelește un apel al unui instrument agent cu semnarea chitanței

Într-o implementare reală, nu dorești ca fiecare autor al agentului să-și amintească să apeleze `make_receipt`. Vrei ca semnarea chitanței să fie automată pentru fiecare invocare a instrumentului.

Iată cel mai simplu model: o clasă wrapper care preia orice funcție apelabilă a instrumentului și returnează o versiune care emite chitanță. Acest lucru se adaptează la orice cadru agent, inclusiv Microsoft Agent Framework (`agent_framework.azure`).

Dacă nu ai configurat un proiect Azure AI Foundry, simularea locală de mai jos demonstrează totuși modelul.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Integrarea cu Microsoft Agent Framework

Wrapper-ul `ReceiptedTool` de mai sus este agnostic față de framework. Pentru a-l folosi într-un agent construit cu Microsoft Agent Framework, înregistrează funcția învelită ca unealtă. O schiță (ai înlocui mock-ul cu o înregistrare reală a unei unelte Azure AI Foundry):

```python
# Pseudodcod care arată forma integrării.
# din agent_framework.azure importă AzureAIProjectAgentProvider
# din azure.identity importă AzureCliCredential
#
# furnizor = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = furnizor.create_agent(
#     instrucțiuni="Ești un agent Contoso Travel ...",
#     unelte=[receipted_lookup],   # unealta învelită, nu funcția brută
# )
# răspuns = agent.run("Găsește zboruri de la Sydney la Los Angeles în iunie.")
#
# # După rulare, fiecare apel al unui instrument făcut de agent are o chitanță semnată:
# lanț_audit = receipted_lookup.receipts
```

Framework-ul agentului nu trebuie să știe nimic despre chitanțe. Semnarea chitanțelor este învelită în jurul uneltei, nu este integrată în framework. Aceasta este modalitatea prin care adaugi proveniență codului agentului existent fără a rescrie agentul.


## Recapitulare și provocare suplimentară

Ai:

- Generat o pereche de chei Ed25519.
- Creat și semnat o chitanță pentru un apel de instrument al agentului.
- Verificat chitanța offline folosind doar cheia publică.
- Modificat o chitanță și ai observat eșecul verificării.
- Creat o secvență de trei chitanțe legate printr-un lanț de hash-uri.
- Modificat mijlocul lanțului și ai observat atât eșecul semnăturii, cât și eșecul legăturii din lanț.
- Învelit o funcție a instrumentului cu semnarea automată a chitanței.

**Provocare suplimentară.** Extinde schema chitanței cu un câmp `request_id` (un UUID pentru urmărire distribuită). Actualizează `make_receipt` pentru a-l include și confirmă că chitanțele se verifică complet. Apoi modifică câmpul după semnare și confirmă că verificarea eșuează. Acest lucru te obligă să conștientizezi cum fiecare byte din codarea canonică contribuie la semnătură.

**Graniță importantă.** Chitanțele dovedesc trei lucruri și doar trei: atribuirea (această cheie a semnat acest conținut), integritatea (conținutul nu s-a schimbat după semnare) și ordinea (această chitanță a venit după acea chitanță). Ele NU dovedesc că acțiunea agentului a fost corectă, că politica denumită în `policy_id` a fost efectiv evaluată sau că agentul a urmat fiecare regulă. Chitanțele sunt o fundație. Guvernanța este sistemul pe care îl construiești deasupra.

Recitește README-ul lecției având această graniță în minte. Cea mai frecventă greșeală pe care o fac echipele cu chitanțele este să presupună că „avem chitanțe” înseamnă „suntem guvernați.” Nu este așa. Chitanțele fac comportamentul agentului auditat. Ele nu îl fac corect.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
